# Intro

https://docs.google.com/document/d/1XGVbEZDa58eGfv7g8V5xEPlrJkVaK-yTW469p9xQvIw/edit?tab=t.0


Pre-processing
1) Pedro - Dialogues are split by line_number, creating an unnecessary row on the dataset
2) Pedro - Multi-Word Names and Title/Honorific Handling
3) ✅ Renan - Quotation Attribution and Implicit Speaker Ambiguity


Visualization
1) Pedro - Speaker Distribution & Frequency - words per scene/act
2) Renan - Scene-Level Interaction Heatmap - which character pairs share the most scenes?

Deliverables
1) Thea - Create PDF
2) Thea - Create presentation
3) All - code (this notebook)

# Setup

In [5]:
%pip install -q --upgrade pip kagglehub
%pip install pandas

import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("umerhaddii/shakespeare-plays-dialogues")

print("Path to dataset files:", path)

# List all CSV files in the dataset
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f'Found {len(csv_files)} CSV files:')
for f in sorted(csv_files):
    size_kb = os.path.getsize(os.path.join(path, f)) / 1024
    print(f'  {f} ({size_kb:.1f} KB)')

# Read romeo_juliet.csv
rj_path = os.path.join(path, 'romeo_juliet.csv')
df = pd.read_csv(rj_path)
print(f'\nShape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Using Colab cache for faster access to the 'shakespeare-plays-dialogues' dataset.
Path to dataset files: /kaggle/input/shakespeare-plays-dialogues
Found 3 CSV files:
  hamlet.csv (289.1 KB)
  macbeth.csv (167.4 KB)
  romeo_juliet.csv (224.6 KB)

Shape: (3282, 5)
Columns: ['act', 'scene', 'character', 'dialogue', 'line_number']


,act,scene,character,dialogue,line_number
0,Act I,Prologue,Chorus,"Two households, both alike in dignity,",1.0
1,Act I,Prologue,Chorus,"In fair Verona, where we lay our scene,",2.0
2,Act I,Prologue,Chorus,"From ancient grudge break to new mutiny,",3.0
3,Act I,Prologue,Chorus,Where civil blood makes civil hands unclean.,4.0
4,Act I,Prologue,Chorus,From forth the fatal loins of these two foes,5.0


# Pre-processing - creates a new "partipants" column - Quotation Attribution and Implicit Speaker Ambiguity

In [6]:
# Pre-processing
# Quotation Attribution and Implicit Speaker Ambiguity
# Creates a new "partipants" column in DataFrame `df`

import re
import html
from typing import Optional
from collections import defaultdict


def add_participants(df: pd.DataFrame, direction_col='character', dialogue_col='dialogue',
                     act_col='act', scene_col='scene', stage_label='[stage direction]') -> pd.DataFrame:
    """
    Add a 'participants' column to a Shakespeare-style dialogue DataFrame.
    Tracks who is physically on-stage by parsing Enter / Exit / Exeunt / Re-enter
    stage directions.  Only characters that actually speak somewhere in the play
    are ever inserted (no invented 'Guests', 'Attendants', etc.).
    """
    # 1. Build a lookup of every real speaker in the corpus
    speakers = df.loc[df[direction_col] != stage_label, direction_col].unique()
    char_map = {}
    for c in speakers:
        key = c.strip().upper()
        char_map[key] = c
        if key.startswith('THE '):
            char_map[key[4:]] = c          # 'The Prince' → 'Prince'

    def _clean(text: str) -> str:
        text = text.strip()
        text = re.sub(r'\s+', ' ', text)
        for pat in [r'^[aA]\s+', r'^[tT][hH][eE]\s+', r'^[aA][nN][dD]\s+',
                    r'^[wW][iI][tT][hH]\s+', r'^[hH][iI][sS]\s+', r'^[hH][eE][rR]\s+']:
            text = re.sub(pat, '', text)
        text = re.sub(r',.*$', '', text)
        text = re.sub(r'\s+(of|with|in|armed|bearing|meeting|booted|disguised|asleep|who)\s+.*$', '',
                      text, flags=re.IGNORECASE)
        text = re.sub(r'\s+(above|below|within|without|booted|disguised|asleep)$', '',
                      text, flags=re.IGNORECASE)
        return text.strip()

    def _extract(text: str) -> list[str]:
        names = []
        for part in re.split(r',\s*|;\s*|\s+and\s+', text):
            cleaned = _clean(part)
            if cleaned and cleaned.upper() in char_map:
                names.append(char_map[cleaned.upper()])
        return names

    def _process_direction(dial: str, present: list, last_speaker: Optional[str]) -> list:
        d = html.unescape(str(dial).strip())
        du = d.upper()

        m = re.search(r'\b(ENTER|RE-ENTER)\b', du)
        if m:
            after = d[m.end():].strip()
            after = re.sub(r'^,\s*', '', after)
            for n in _extract(after):
                if n not in present:
                    present.append(n)
            return present

        if du.startswith('EXEUNT'):
            after = re.sub(r'^EXEUNT\s*', '', d, flags=re.IGNORECASE).strip()
            if not after or after in ('.', ','):
                present = []
            elif re.match(r'^all\s+but\s+', after, flags=re.IGNORECASE):
                after = re.sub(r'^all\s+but\s+', '', after, flags=re.IGNORECASE)
                keep = _extract(after)
                present = [n for n in keep if n in present]
            else:
                for n in _extract(after):
                    if n in present:
                        present.remove(n)
            return present

        if du.startswith('EXIT'):
            after = re.sub(r'^EXIT\s*', '', d, flags=re.IGNORECASE).strip()
            if not after or after in ('.', ','):
                if last_speaker and last_speaker in present:
                    present.remove(last_speaker)
            else:
                for n in _extract(after):
                    if n in present:
                        present.remove(n)
            return present

        return present

    # 2. Walk the DataFrame row-by-row
    participants = []
    present = []
    last_speaker = None
    cur_act, cur_scene = None, None

    for _, row in df.iterrows():
        act, scene = row[act_col], row[scene_col]
        char = row[direction_col]
        dial = row[dialogue_col]

        if (act, scene) != (cur_act, cur_scene):
            present = []
            last_speaker = None
            cur_act, cur_scene = act, scene

        if char == stage_label:
            present = _process_direction(dial, present, last_speaker)
        else:
            if char not in present:
                present.append(char)
            last_speaker = char

        participants.append(list(present))

    return df.assign(participants=participants)


# --- Run it ---
df = add_participants(df)

# --- Quick sanity checks ---
print('Rows:', len(df))
print('Columns:', list(df.columns))

df.head(100)


Rows: 3282
Columns: ['act', 'scene', 'character', 'dialogue', 'line_number', 'participants']


,act,scene,character,dialogue,line_number,participants
0,Act I,Prologue,Chorus,"Two households, both alike in dignity,",1.0,[Chorus]
1,Act I,Prologue,Chorus,"In fair Verona, where we lay our scene,",2.0,[Chorus]
2,Act I,Prologue,Chorus,"From ancient grudge break to new mutiny,",3.0,[Chorus]
3,Act I,Prologue,Chorus,Where civil blood makes civil hands unclean.,4.0,[Chorus]
4,Act I,Prologue,Chorus,From forth the fatal loins of these two foes,5.0,[Chorus]
...,...,...,...,...,...,...
95,Act I,Scene I,Montague,"Thou villain Capulet,--Hold me not, let me go.",86.0,"[Sampson, Gregory, Abraham, Balthasar, Benvoli..."
96,Act I,Scene I,Lady Montague,Thou shalt not stir a foot to seek a foe.,87.0,"[Sampson, Gregory, Abraham, Balthasar, Benvoli..."
97,Act I,Scene I,[stage direction],"Enter PRINCE, with Attendants",NaN,"[Sampson, Gregory, Abraham, Balthasar, Benvoli..."
98,Act I,Scene I,Prince,"Rebellious subjects, enemies to peace,",88.0,"[Sampson, Gregory, Abraham, Balthasar, Benvoli..."


In [7]:
def add_participants_enhanced(df: pd.DataFrame,
                              direction_col='character',
                              dialogue_col='dialogue',
                              act_col='act',
                              scene_col='scene',
                              stage_label='[stage direction]') -> pd.DataFrame:
    """
    Replacement of add_participants with improved multi-word name and title/honorific handling.
    """
    import re, html
    from typing import Optional

    # normalize names to compact uppercase keys
    def _normalize_name(text: str) -> str:
        if not isinstance(text, str):
            return ''
        t = text.upper()
        t = re.sub(r'\(.*?\)', ' ', t)               # remove parentheticals
        t = re.sub(r'[^A-Z0-9\s\-\'\.]', ' ', t)     # keep useful chars
        t = re.sub(r'\s+', ' ', t).strip()
        t = t.strip('.')
        return t

    TITLES = {'LORD','LADY','SIR','MADAM','MRS','MR','MISTRESS','PRINCE','KING','QUEEN',
              'DUKE','EARL','COUNT','COUNTESS','FATHER','MOTHER','CAPTAIN','DOCTOR','DR','GENTLEMAN'}

    # build enhanced char_map from actual speakers
    speakers = df.loc[df[direction_col] != stage_label, direction_col].dropna().unique()
    char_map: dict[str,str] = {}
    for c in speakers:
        if not isinstance(c, str) or not c.strip():
            continue
        norm = _normalize_name(c)
        if not norm:
            continue
        char_map[norm] = c
        # variants: drop leading THE
        if norm.startswith('THE '):
            char_map[norm[4:].strip()] = c
        # strip a single leading title
        parts = norm.split()
        if parts and parts[0] in TITLES:
            without_title = ' '.join(parts[1:]).strip()
            if without_title:
                char_map[without_title] = c
        # last-name variant
        if parts:
            last = parts[-1]
            if last:
                char_map[last] = c
        # first+last for multi-word names
        if len(parts) >= 2:
            fl = parts[0] + ' ' + parts[-1]
            char_map[fl] = c

    # Small helper to clean a segment from a stage direction or list of names
    def _clean(text: str) -> str:
        if not isinstance(text, str):
            return ''
        text = text.strip()
        text = re.sub(r'\s+', ' ', text)
        for pat in [r'^[aA]\s+', r'^[tT][hH][eE]\s+', r'^[aA][nN][dD]\s+',
                    r'^[wW][iI][tT][hH]\s+', r'^[hH][iI][sS]\s+', r'^[hH][eE][rR]\s+']:
            text = re.sub(pat, '', text)
        text = re.sub(r',.*$', '', text)
        text = re.sub(r'\s+(of|with|in|armed|bearing|meeting|booted|disguised|asleep|who)\s+.*$',
                      '', text, flags=re.IGNORECASE)
        text = re.sub(r'\s+(above|below|within|without|booted|disguised|asleep)$', '',
                      text, flags=re.IGNORECASE)
        return text.strip()

    # improved extractor using normalized lookup
    def _extract(text: str) -> list[str]:
        names: list[str] = []
        for part in re.split(r',\s*|;\s*|\s+and\s+', str(text)):
            cleaned = _clean(part)
            key = _normalize_name(cleaned)
            if key and key in char_map:
                names.append(char_map[key])
        return names

    # reuse original direction processor but using enhanced _extract
    def _process_direction(dial: str, present: list, last_speaker: Optional[str]) -> list:
        d = html.unescape(str(dial).strip())
        du = d.upper()

        m = re.search(r'\b(ENTER|RE-ENTER)\b', du)
        if m:
            after = d[m.end():].strip()
            after = re.sub(r'^,\s*', '', after)
            for n in _extract(after):
                if n not in present:
                    present.append(n)
            return present

        if du.startswith('EXEUNT'):
            after = re.sub(r'^EXEUNT\s*', '', d, flags=re.IGNORECASE).strip()
            if not after or after in ('.', ','):
                present = []
            elif re.match(r'^all\s+but\s+', after, flags=re.IGNORECASE):
                after = re.sub(r'^all\s+but\s+', '', after, flags=re.IGNORECASE)
                keep = _extract(after)
                present = [n for n in keep if n in present]
            else:
                for n in _extract(after):
                    if n in present:
                        present.remove(n)
            return present

        if du.startswith('EXIT'):
            after = re.sub(r'^EXIT\s*', '', d, flags=re.IGNORECASE).strip()
            if not after or after in ('.', ','):
                if last_speaker and last_speaker in present:
                    present.remove(last_speaker)
            else:
                for n in _extract(after):
                    if n in present:
                        present.remove(n)
            return present

        return present

    # walk rows (same logic as original)
    participants = []
    present: list[str] = []
    last_speaker = None
    cur_act, cur_scene = None, None

    for _, row in df.iterrows():
        act, scene = row[act_col], row[scene_col]
        char = row[direction_col]
        dial = row[dialogue_col]

        if (act, scene) != (cur_act, cur_scene):
            present = []
            last_speaker = None
            cur_act, cur_scene = act, scene

        if char == stage_label:
            present = _process_direction(dial, present, last_speaker)
        else:
            # normalize speaker name when adding to present (map variants to canonical speaker)
            if isinstance(char, str):
                key = _normalize_name(char)
                canon = char_map.get(key, char)
            else:
                canon = char
            if canon not in present:
                present.append(canon)
            last_speaker = canon

        participants.append(list(present))

    return df.assign(participants=participants)

df.head(100)

,act,scene,character,dialogue,line_number,participants
0,Act I,Prologue,Chorus,"Two households, both alike in dignity,",1.0,[Chorus]
1,Act I,Prologue,Chorus,"In fair Verona, where we lay our scene,",2.0,[Chorus]
2,Act I,Prologue,Chorus,"From ancient grudge break to new mutiny,",3.0,[Chorus]
3,Act I,Prologue,Chorus,Where civil blood makes civil hands unclean.,4.0,[Chorus]
4,Act I,Prologue,Chorus,From forth the fatal loins of these two foes,5.0,[Chorus]
...,...,...,...,...,...,...
95,Act I,Scene I,Montague,"Thou villain Capulet,--Hold me not, let me go.",86.0,"[Sampson, Gregory, Abraham, Balthasar, Benvoli..."
96,Act I,Scene I,Lady Montague,Thou shalt not stir a foot to seek a foe.,87.0,"[Sampson, Gregory, Abraham, Balthasar, Benvoli..."
97,Act I,Scene I,[stage direction],"Enter PRINCE, with Attendants",NaN,"[Sampson, Gregory, Abraham, Balthasar, Benvoli..."
98,Act I,Scene I,Prince,"Rebellious subjects, enemies to peace,",88.0,"[Sampson, Gregory, Abraham, Balthasar, Benvoli..."


# Pre-processing - Multi-Word Names and Title/Honorific Handling

# Pre-processing - Dialogues are split by line_number, creating an unnecessary row on the dataset

# Data visualization - Speaker Distribution & Frequency - words per scene/act

# Data visualization - Character Interaction Heatmap